# 02 — Medical NER Experiments

Compare **BioBERT**, **ClinicalBERT**, and **SciSpacy** on medical entity recognition.

## What this notebook covers
- Load and run each NER model on sample medical text
- Compute Precision / Recall / F1 per entity type
- Visualise confusion matrices and entity frequency
- Export comparison table


In [ ]:
import sys
sys.path.insert(0, '..')  # run from notebooks/

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rich import print as rprint

from src.extraction.ner_extractor import MedicalNERPipeline, BioBERTExtractor, SciSpacyExtractor
from src.evaluation.evaluator import NEREvaluator
from src.utils.config import get_settings

cfg = get_settings('../configs/config.yaml')
print('Config loaded ✓')

## 1. Sample Medical Texts

In [ ]:
TEST_TEXTS = [
    {
        'text': 'Patient has hemoglobin 10.5 g/dL indicating iron deficiency anemia. '
                'Started on ferrous sulfate 325 mg twice daily.',
        'entities': [
            {'text': 'hemoglobin', 'label': 'LAB_TEST'},
            {'text': 'iron deficiency anemia', 'label': 'DISEASE'},
            {'text': 'ferrous sulfate', 'label': 'MEDICATION'},
            {'text': '325 mg', 'label': 'DOSAGE'},
        ]
    },
    {
        'text': 'Fasting glucose 126 mg/dL, HbA1c 7.2%. Diagnosis: Type 2 diabetes. '
                'Prescribed metformin 500 mg BID.',
        'entities': [
            {'text': 'glucose', 'label': 'LAB_TEST'},
            {'text': 'HbA1c', 'label': 'LAB_TEST'},
            {'text': 'Type 2 diabetes', 'label': 'DISEASE'},
            {'text': 'metformin', 'label': 'MEDICATION'},
        ]
    },
    {
        'text': 'Elevated total cholesterol at 245 mg/dL with LDL 175 mg/dL. '
                'HDL 35 mg/dL. Patient advised statin therapy and dietary modification.',
        'entities': [
            {'text': 'total cholesterol', 'label': 'LAB_TEST'},
            {'text': 'LDL', 'label': 'LAB_TEST'},
            {'text': 'HDL', 'label': 'LAB_TEST'},
        ]
    },
]
print(f'Loaded {len(TEST_TEXTS)} test samples')

## 2. Run BioBERT NER

In [ ]:
pipeline = MedicalNERPipeline(cfg.extraction)

results = []
for sample in TEST_TEXTS:
    result = pipeline.run(sample['text'])
    results.append(result)
    rprint(f'\n[bold]Text:[/bold] {sample["text"][:80]}…')
    rprint(f'[bold]Entities found:[/bold] {result.entity_summary}')
    rprint(f'[bold]Lab values:[/bold] {result.lab_values}')

## 3. Evaluate NER Metrics

In [ ]:
evaluator = NEREvaluator()
predictions = [[e.to_dict() for e in r.entities] for r in results]
ground_truth = [s['entities'] for s in TEST_TEXTS]

metrics = evaluator.evaluate(predictions, ground_truth)

df = pd.DataFrame([vars(m) for m in metrics])
df = df.set_index('entity_type')
print(df.to_string())

## 4. Visualise F1 Scores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 per entity type
df['f1'].sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('F1 Score by Entity Type')
axes[0].set_xlabel('F1')
axes[0].axvline(x=0.7, color='red', linestyle='--', alpha=0.5, label='0.7 threshold')
axes[0].legend()

# Precision vs Recall scatter
axes[1].scatter(df['precision'], df['recall'], s=df['support']*20+20, alpha=0.7, c='steelblue')
for label, row in df.iterrows():
    axes[1].annotate(label, (row['precision'], row['recall']), fontsize=8, ha='center', va='bottom')
axes[1].set_xlabel('Precision')
axes[1].set_ylabel('Recall')
axes[1].set_title('Precision vs Recall (bubble size = support)')
axes[1].set_xlim(0, 1.1)
axes[1].set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('../outputs/ner_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to outputs/ner_metrics.png')

## 5. Entity Frequency Analysis

In [ ]:
from collections import Counter

all_entities = []
for r in results:
    for ent in r.entities:
        all_entities.append({'text': ent.text, 'label': ent.label})

ent_df = pd.DataFrame(all_entities)
if not ent_df.empty:
    label_counts = ent_df['label'].value_counts()
    label_counts.plot(kind='pie', autopct='%1.1f%%', figsize=(8, 6), title='Entity Type Distribution')
    plt.savefig('../outputs/entity_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\nTop entities by label:')
    for label in ent_df['label'].unique():
        top = ent_df[ent_df['label']==label]['text'].value_counts().head(3)
        print(f'  {label}: {list(top.index)}')